In [1]:
import datetime
import calendar
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
import datetime
import unicodedata

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el mes anterior
if hoy.month == 1:
    mes_anterior = 12 
    anio_anterior = hoy.year - 1
else:
    mes_anterior = hoy.month -1
    anio_anterior = hoy.year

meses_esp = [
    "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
    "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]

# Asignar variables con el mes anterior
MES_A_LIQUIDAR = f"{mes_anterior}/{anio_anterior}"
MES_RECAUDO_NUM = mes_anterior
PERIODO_FINANCIERO = f"{anio_anterior}{mes_anterior:02d}"
MES_FACTURAR = meses_esp[mes_anterior - 1] 

# Calcular el mes anterior
if hoy.month == 1:
    mes_anterior_2 = 12 - 2
    anio_anterior_2 = hoy.year 
else:
    mes_anterior_2 = hoy.month - 2
    anio_anterior_2 = hoy.year  

meses_esp = [
    "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
    "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]

# Asignar variables con el mes anterior
MES_A_LIQUIDAR_ANTERIOR = f"{mes_anterior_2}/{anio_anterior_2}"
MES_ARCHIVO_EXCEL = f"{MES_FACTURAR}_{anio_anterior_2}"

# Imprimir resultados
print("MES_A_LIQUIDAR =", MES_A_LIQUIDAR)
print("MES_ANTERIOR_A_LIQUIDAR =", MES_A_LIQUIDAR_ANTERIOR)
print("MES_RECAUDO_NUM =", MES_RECAUDO_NUM)
print("PERIODO_FINANCIERO =", PERIODO_FINANCIERO)
print("MES_FACTURAR =", MES_FACTURAR)
print("MES_ARCHIVO_EXCEL =", MES_ARCHIVO_EXCEL)  

MES_A_LIQUIDAR = 9/2026
MES_ANTERIOR_A_LIQUIDAR = 8/2026
MES_RECAUDO_NUM = 9
PERIODO_FINANCIERO = 202609
MES_FACTURAR = SEPTIEMBRE
MES_ARCHIVO_EXCEL = SEPTIEMBRE_2026


In [3]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
                    SELECT * FROM CUN_REPOSITORIO.AGUEVARA.VENTAS_NOVEDADES_TELECAMPUS
                    where ESTADO_FINAL = 'VALIDO'
             ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_ventas = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_ventas.shape        

Conexion a la base de datos realizada


(2744, 16)

In [4]:
df_ventas.columns

Index(['DOC_ALUM', 'PERIODO', 'NUEVO', 'CLASE_ACTUAL', 'FUERZAZOHO',
       'ESTADO_PAGO', 'EST_MATRICULADO', 'TIPO', 'NOM_PROGRAMA',
       'PROPIETARIO_ZOHO', 'LLAVE', 'ESTADO_FINAL', 'VINCULACIONES', 'CONTACT',
       'OPORTUNIDADES', 'PERIODO_ACTIVO'],
      dtype='object')

In [5]:
df_recaudos_temp1=df_ventas[['DOC_ALUM', 'PERIODO', 'NUEVO', 'CLASE_ACTUAL', 'FUERZAZOHO',
       'ESTADO_PAGO', 'EST_MATRICULADO', 'TIPO', 'NOM_PROGRAMA',
       'PROPIETARIO_ZOHO', 'LLAVE']].copy()
df_recaudos_temp1 = df_recaudos_temp1.drop_duplicates()
df_recaudos_temp1['DOC_ALUM'] = df_recaudos_temp1['DOC_ALUM'].astype(str)

In [6]:
para_orden = df_recaudos_temp1['DOC_ALUM'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
periodos_unicos = df_recaudos_temp1['PERIODO'].unique()
valores_unicos_periodos = "', '".join(map(str, periodos_unicos))
valores_unicos_periodos = f"'{valores_unicos_periodos}'"

# SE TRAEN LOS RECAUDOS

In [7]:
from sqlalchemy import create_engine, text
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'
driver = 'ODBC Driver 17 for SQL Server'  # Asegúrate de tenerlo instalado


# ----------------------------- #
# Crear la cadena de conexión
# ----------------------------- #
connection_url = (
    f"mssql+pyodbc://@{host}:{port}/{database}"
    "?trusted_connection=yes"
    f"&driver={driver.replace(' ', '+')}"
)

# ----------------------------- #
# Ejecutar la consulta
# ----------------------------- #
try:
    engine = create_engine(connection_url)
    print("Conexión a la base de datos realizada")

    sql_query = text(f'''
        SELECT t.*, 
               COALESCE(orden_relacionada, orden) AS orden_final
        FROM openquery([172.16.1.175], '
            SELECT r.*, 
                   t.documento AS documento_relacionado, 
                   t.orden AS orden_relacionada,
                   ''ICETEX'' AS tipo_pago
            FROM iceberg.r_recaudo_pagos r
            LEFT JOIN ICEBERG.cunt_tramite_externo t 
                   ON t.periodo = r.periodo
                   AND t.identificacion = r.doc_alum
                   AND t.orden_inicial = r.orden
                   AND t.documento_inicial = r.documento
            WHERE (r.fecha_de_pago like ''%2026%'')
              AND r.FECHA_DE_PAGO_ICETEX LIKE ''%2026%''  
              AND R.VALOR_PAGADO_ICETEX IS NOT NULL

            UNION ALL

            SELECT r.*, 
                   t.documento, 
                   t.orden,
                   ''CREDITY'' AS tipo_pago
            FROM iceberg.r_recaudo_pagos r
            LEFT JOIN ICEBERG.cunt_tramite_externo t 
                   ON t.periodo = r.periodo
                   AND t.identificacion = r.doc_alum
                   AND t.orden_inicial = r.orden
                   AND t.documento_inicial = r.documento
            WHERE (r.fecha_de_pago like ''%2026%'')
              AND r.FECHA_DE_PAGO_CREDITY LIKE ''%2026%'' 
              AND R.VALOR_PAGADO_CREDITY IS NOT NULL

            UNION ALL

            SELECT r.*, 
                   t.documento, 
                   t.orden,
                   ''FINCOMERCIO'' AS tipo_pago
            FROM iceberg.r_recaudo_pagos r
            LEFT JOIN ICEBERG.cunt_tramite_externo t 
                   ON t.periodo = r.periodo
                   AND t.identificacion = r.doc_alum 
                   AND t.orden_inicial = r.orden
                   AND t.documento_inicial = r.documento
            WHERE (r.fecha_de_pago like ''%2026%'')
              AND r.FECHA_DE_PAGO_FINCOMERCIO LIKE ''%2026%''  
              AND R.VALOR_PAGADO_FINCOMERCIO IS NOT NULL

            UNION ALL

            SELECT r.*, 
                   t.documento, 
                   t.orden,
                   ''Normal'' AS tipo_pago
            FROM iceberg.r_recaudo_pagos r
            LEFT JOIN ICEBERG.cunt_tramite_externo t 
                   ON t.periodo = r.periodo
                   AND t.identificacion = r.doc_alum
                   AND t.orden_inicial = r.orden
                   AND t.documento_inicial = r.documento
            WHERE (r.fecha_de_pago like ''%2026%'')
              AND  r.FECHA_DE_PAGO LIKE ''%2026%'' 
        ') t
        WHERE doc_alum IN ({valores_unicos_df1_orden})
          AND periodo IN ({valores_unicos_periodos})
          AND PERIODO not like '%P'
          AND PERIODO NOT LIKE '%I%'
    ''')

    # Ejecutar consulta y guardar en DataFrame
    with engine.connect() as connection:
        df_r_recaudos = pd.read_sql_query(sql_query, connection)

except Exception as e:
    print("Error:", e)

finally:
    if 'engine' in locals():
        engine.dispose()
        print("Conexión cerrada")
df_r_recaudos.shape

Conexión a la base de datos realizada
Conexión cerrada


(630, 39)

In [8]:
df_r_recaudos_copy_copy = df_r_recaudos.copy()

In [9]:
df_r_recaudos_copy_copy['VALOR_INGRESO_DIRECTO'] = df_r_recaudos_copy_copy['VALOR_INGRESO_DIRECTO'].astype(int)

def ajustar_pago(df):
    # Asegurar que los NaN en valor directo sean 0
    df['VALOR_INGRESO_DIRECTO'] = df['VALOR_INGRESO_DIRECTO'].fillna(0)

    # Crear la columna de FECHA_PAGO si no existe
    if 'FECHA_PAGO' not in df.columns:
        df['FECHA_PAGO'] = pd.NaT

    # ICETEX
    icetex_mask = (df['VALOR_INGRESO_DIRECTO'] == 0) & (df['VALOR_PAGADO_ICETEX'].notna())
    df.loc[icetex_mask, 'VALOR_INGRESO_DIRECTO'] = df.loc[icetex_mask, 'VALOR_PAGADO_ICETEX']
    df.loc[icetex_mask, 'FECHA_PAGO'] = df.loc[icetex_mask, 'FECHA_DE_PAGO_ICETEX']

    # CREDITY
    credity_mask = (
        (df['VALOR_INGRESO_DIRECTO'] == 0) &
        (df['VALOR_PAGADO_ICETEX'].isna()) &
        (df['VALOR_PAGADO_CREDITY'].notna())
    )
    df.loc[credity_mask, 'VALOR_INGRESO_DIRECTO'] = df.loc[credity_mask, 'VALOR_PAGADO_CREDITY']
    df.loc[credity_mask, 'FECHA_PAGO'] = df.loc[credity_mask, 'FECHA_DE_PAGO_CREDITY']

    # FINCOMERCIO
    finco_mask = (
        (df['VALOR_INGRESO_DIRECTO'] == 0) &
        (df['VALOR_PAGADO_ICETEX'].isna()) &
        (df['VALOR_PAGADO_CREDITY'].isna()) &
        (df['VALOR_PAGADO_FINCOMERCIO'].notna())
    )
    df.loc[finco_mask, 'VALOR_INGRESO_DIRECTO'] = df.loc[finco_mask, 'VALOR_PAGADO_FINCOMERCIO']
    df.loc[finco_mask, 'FECHA_PAGO'] = df.loc[finco_mask, 'FECHA_DE_PAGO_FINCOMERCIO']

    return df
df_r_recaudos_copy_copy = ajustar_pago(df_r_recaudos_copy_copy)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_24904\1945575953.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[icetex_mask, 'VALOR_INGRESO_DIRECTO'] = df.loc[icetex_mask, 'VALOR_PAGADO_ICETEX']


In [10]:
df_r_recaudos_copy_copy['FECHA_PAGO'] = pd.to_datetime(df_r_recaudos_copy_copy['FECHA_PAGO'], errors='coerce')
df_r_recaudos_copy_copy['FECHA_PAGO'] = df_r_recaudos_copy_copy['FECHA_PAGO'].dt.strftime('%d/%m/%Y')


In [11]:
df_r_recaudos_copy_copy['FECHA_PAGO'] = np.where( df_r_recaudos_copy_copy['FECHA_PAGO'].isna() & 
        df_r_recaudos_copy_copy['FECHA_DE_PAGO'].notna(), df_r_recaudos_copy_copy['FECHA_DE_PAGO'],
        df_r_recaudos_copy_copy['FECHA_PAGO'])
df_r_recaudos_copy_copy.shape

(630, 40)

In [62]:
df_r_recaudos_copy_copy

,TIP_IDENTIFICACION,DOC_ALUM,ORGANIZACION,NOMBRE_ESTUDIANTE,CORREO_INSTITUCIONAL,CORREO_PERSONAL,TELEFONO_FIJO,TELEFONO_MOVIL,PERIODO,DOCUMENTO,...,FECHA_DE_PAGO,NUMERO,CAJA,DESCRIPCION,DOCUMENTO_RELACIONADO,ORDEN_RELACIONADA,TIPO_PAGO,orden_final,FECHA_PAGO,FECHA_FILTRO
0,C,1000120154,1,VIRGUEZ MERIZALDE YISSELA YIRLEY,yissela.virguez@cun.edu.co,yisselavirguez28@gmail.com,3115890411,3124675351,2026A,FAMA,...,26/03/2026,357124962.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3320336.0,26/03/2026,2026-03-26
1,C,1000786771,1,RAMIREZ HERRERA WILLMAN ALEXANDER,willman.ramirez@cun.edu.co,willmanalexander20@gmail.com,3142887394,3142887394,2026A,FAMA,...,27/01/2026,339564049.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3309451.0,27/01/2026,2026-01-27
2,C,1019002299,1,PALOMINO CHACON DIEGO ANDREY,diego.palomino@cun.edu.co,palominodiegoandrey@gmail.com,3229202317,3229202317,2026A,FAMA,...,15/04/2026,359568082.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3317197.0,15/04/2026,2026-04-15
3,T,1091969757,1,URIBE LOPEZ SHEYLA YULIETH,sheyla.uribe@cun.edu.co,sheylalopez1982@gmail.com,3006941976,3006941976,2026A,FAMA,...,23/03/2026,356724916.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3388424.0,23/03/2026,2026-03-23
4,C,1016105862,1,MAYORGA GUZMAN OSCAR SANTIAGO,oscar.mayorgag@cun.edu.co,santi.mayo9820@gmail.com,3125743143,3125743143,25V06,FAMA,...,13/04/2026,359450123.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3190292.0,13/04/2026,2026-04-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625,C,1032392690,1,VALENCIA GOMEZ YENCY TATIANA,yency.valencia@cun.edu.co,tatis_vgomez@hotmail.com,172414407,172414407,2026A,FAMA,...,09/01/2026,333378855.0,201,Pago automatico por Boton de Pagos. Referencia...,None,NaN,Normal,3320244.0,09/01/2026,2026-01-09
626,C,1001445835,1,MONTOYA MANUEL ALEJANDRO,manuel.montoya@cun.edu.co,montoyamanuel2003@gmail.com,3209538930,3209538930,2025C,FAMA,...,24/04/2026,360371850.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,2978761.0,24/04/2026,2026-04-24
627,C,1019996209,1,ARIAS CASTELBLANCO JUAN ESTEBAN,juan.ariascas@cun.edu.co,juanestebanarias.1019@gmail.com,3005595509,3005595509,25V06,FAMA,...,31/01/2026,340975894.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3221291.0,31/01/2026,2026-01-31
628,C,1066575226,1,CARE MEJIA DEIVIS HANS,deivis.care@cun.edu.co,dcare9474@gmail.com,3242619060,3242619060,25V05,FAMA,...,30/04/2026,360919357.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3160881.0,30/04/2026,2026-04-30


In [13]:
df_r_recaudos_copy_copy['FECHA_FILTRO'] = pd.to_datetime(
    df_r_recaudos_copy_copy['FECHA_PAGO'],
    format='%d/%m/%Y',
    errors='coerce'
)

# 2. Convertir los meses en fechas
fecha_min = pd.to_datetime(
    '01/' + MES_A_LIQUIDAR_ANTERIOR,
    format='%d/%m/%Y'
)

# Obtener automáticamente el último día del mes
fecha_max = (
    pd.to_datetime('01/' + MES_A_LIQUIDAR, format='%d/%m/%Y')
    + pd.offsets.MonthEnd(0)
)

# 3. Filtrar entre las fechas
df_r_recaudos_copy = df_r_recaudos_copy_copy[
    (df_r_recaudos_copy_copy['FECHA_FILTRO'] >= fecha_min) &
    (df_r_recaudos_copy_copy['FECHA_FILTRO'] <= fecha_max)
].copy()

df_r_recaudos_copy.shape

(7, 41)

In [14]:
df_r_recaudos_copy

,TIP_IDENTIFICACION,DOC_ALUM,ORGANIZACION,NOMBRE_ESTUDIANTE,CORREO_INSTITUCIONAL,CORREO_PERSONAL,TELEFONO_FIJO,TELEFONO_MOVIL,PERIODO,DOCUMENTO,...,FECHA_DE_PAGO,NUMERO,CAJA,DESCRIPCION,DOCUMENTO_RELACIONADO,ORDEN_RELACIONADA,TIPO_PAGO,orden_final,FECHA_PAGO,FECHA_FILTRO
33,C,1.034396e+09,1,SARMIENTO LOPEZ LAURA SOFÍA,laura.sarmiento@cun.edu.co,sarmienlol21@gmail.com,169200104,3169200104,2025D,FAMA,...,03/08/2026,371862506.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3157394.0,03/08/2026,2026-08-03
301,C,1.152453e+09,1,GALEANO GÓMEZ SANTIAGO ARLEY,santiago.galeanogom@cun.edu.co,santigaleano19@gmail.com,3332410769,3332410769,25V01,FAMA,...,17/08/2026,373397247.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,2846769.0,17/08/2026,2026-08-17
316,C,8.021348e+07,1,ARIZA GOMEZ FREDY ERNESTO,fredy.arizag@cun.edu.co,fredyariza2009@gmail.com,4621980,3108525359,24ET3,None,...,25/08/2026,374409330.0,200,RECIBO AUTOMATICO CREADO EN EL REGISTRO DEL RE...,None,NaN,Normal,NaN,25/08/2026,2026-08-25
443,C,1.022995e+09,1,ROJAS ARIAS JOHN ALEXANDER,john.rojasa@cun.edu.co,rojas94alexander@gmail.com,3212643314,3212643314,25ES2,FAMA,...,14/08/2026,374658776.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3061370.0,14/08/2026,2026-08-14
510,C,1.013115e+09,1,DIAZ CHICUASUQUE SERGIO,sergio.diazc@cun.edu.co,sergio.diazchicua@gmail.com,3016539885,3016539885,25V05,FAMA,...,06/08/2026,372253859.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3161375.0,06/08/2026,2026-08-06
579,C,1.141318e+09,1,GIL GARZON MILDRED ELIANA,mildred.gil@cun.edu.co,mildredgiljs@gmail.com,3002439651,3002439651,25V06,FAMA,...,20/08/2026,373720288.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3196911.0,20/08/2026,2026-08-20
608,C,1.012424e+09,1,ESCOBAR PRIETO ANGIE LORENA,angie.escobarpri@cun.edu.co,angie.salo.348@gmail.com,3134486432,3134486432,25V06,FAMA,...,05/09/2026,375908517.0,201,Recibo Automatico por Pago en BOTON DE PAGOS d...,None,NaN,Normal,3213048.0,05/09/2026,2026-09-05


In [15]:
df_r_recaudos_copy['FECHA_PAGO'].unique()

array(['03/08/2026', '17/08/2026', '25/08/2026', '14/08/2026',
       '06/08/2026', '20/08/2026', '05/09/2026'], dtype=object)

In [16]:
df_r_recaudos_copy = df_r_recaudos_copy.rename(columns={'PERIODO':'PERIODO_RECAUDO'})
df_r_recaudos_copy = df_r_recaudos_copy[['DOC_ALUM','PERIODO_RECAUDO','VALOR_INGRESO_DIRECTO', 'FECHA_PAGO', 'orden_final']]
df_r_recaudos_copy.shape

(7, 5)

# SE TRAEN LA ORDEN, EL VALOR, ADICIONALES

In [17]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
                    Select DISTINCT DOC_ALUM,COD_UNI,PERIODO,TIPO AS PRODUCTO,VALOR_ORDEN_IC AS valor_orden, 
                         VAL_LIQUIDADO, RECARGO,ADICIONALES,val_becdtos,NCR_ANULACION                                
                    from ZOHO.Base_Personas
                    where DOC_ALUM IN ({valores_unicos_df1_orden})
                    AND ESTADO_PAGO = 'PAGO'
                    AND EST_MATRICULADO = '1-Activo'
                    AND PERIODO IN ({valores_unicos_periodos})
                    AND (DOCUMENTO = 'FAMA' OR DOCUMENTO = 'FECO')
                   --AND ANIO like '%2026%'
                    ORDER BY PERIODO DESC
             ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_recibos_contact = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_recibos_contact.shape

Conexion a la base de datos realizada


(5290, 10)

In [18]:
recibos_contact = df_recibos_contact.copy()

In [19]:
recibos_contact['valor_ingles'] = '0'
recibos_contact['PRODUCTO'] = recibos_contact['PRODUCTO'].str.upper()

mask = recibos_contact['PRODUCTO'] == 'PREGRADO'
recibos_contact.loc[mask, "valor_ingles"] = '200000'

In [20]:
recibos_contact = recibos_contact.drop_duplicates()
recibos_contact['LLAVE'] = recibos_contact['DOC_ALUM'].astype(str) + recibos_contact['PERIODO']
recibos_contact = recibos_contact.drop(columns=['DOC_ALUM','PERIODO'])

In [21]:
validos_recibos_contact = df_recaudos_temp1.merge(recibos_contact,how='inner',on='LLAVE')
validos_recibos_contact.drop_duplicates(subset='LLAVE',inplace=True)
validos_recibos_contact.shape

(2744, 20)

In [22]:
df_r_recaudos_copy['DOC_ALUM'] = df_r_recaudos_copy['DOC_ALUM'].astype(float).astype(int).astype(str)
recibos_contact_1 = df_r_recaudos_copy.merge(validos_recibos_contact,how='inner',on='DOC_ALUM')
recibos_contact_1.drop_duplicates(subset=['DOC_ALUM','PERIODO_RECAUDO','VALOR_INGRESO_DIRECTO','FECHA_PAGO'], inplace=True)
recibos_contact_1 = recibos_contact_1.rename(columns={'FECHA_PAGO':'FECHA_RECAUDO', 'orden_final':'ORDEN','VALOR_INGRESO_DIRECTO':'VALOR_RECAUDO'})
recibos_contact_1.shape

(7, 24)

In [23]:
recibos_contact_1['ORDEN'] = recibos_contact_1['ORDEN'].replace('', 0).fillna(0)
recibos_contact_1['ORDEN']=recibos_contact_1['ORDEN'].astype(float).astype(int).astype(str)
recibos_contact_1['valor_orden'] = recibos_contact_1['valor_orden'].fillna(0)


# SE TRAE TARIFAS DE COMISION Y TELECAMPUS ACTIVOS

In [24]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    tarifas_df = pd.read_sql(f'''
    Select TELECAMPUS,TIPO,PRODUCTO,Por_COMISION1,Por_COMISION1_ING
    from CUN_REPOSITORIO.Comisiones.Tarifas_Telecampus_MODELO3
                ''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [25]:

tarifas_df['TELECAMPUS'] = tarifas_df['TELECAMPUS'].apply(
    lambda x: unicodedata.normalize('NFKD', str(x))
    .encode('ascii', 'ignore')
    .decode('utf-8')
    if pd.notna(x) else x
)
tarifas_df['TELECAMPUS'] = tarifas_df['TELECAMPUS'].str.upper()

In [26]:
tarifas_df['TELECAMPUS'] = tarifas_df['TELECAMPUS'].apply(
    lambda x: x if 'TELECAMPUS' in x else f'TELECAMPUS {x}'
)

In [27]:
lista_telecampus = (
    tarifas_df['TELECAMPUS']
    .str.replace('TELECAMPUS ', '', regex=False)
    .unique()
)
lista_telecampus

array(['TOLIMA', 'BOSA', 'KENNEDY', 'MEDELLIN', 'SANTA MARTA', 'SUBA',
       'UNION', 'BELLO', 'TUNJA', 'IBAGUE', 'ENGATIVA', 'ARJONA',
       'ARMENIA', 'CERETE CORDOBA', 'BARRANQUILLA SOLEDAD', 'NEIVA',
       'BARRANQUILLA', 'BUCARAMANGA', 'CARTAGENA', 'MOSQUERA', 'GUAVIO',
       'POLIMODAL', 'MONTERIA', 'SINCELEJO', 'SOTAVENTO', 'VILLAVICENCIO',
       'BARRANQUILLA CENTRO', 'CALI CENTRO', 'CUCUTA', 'FONTANAR',
       'CIUDAD BOLIVAR', 'MONTERIA CENTRO', 'DUITAMA', 'HUILA',
       'MANIZALES', 'PEREIRA', 'USME', 'TAME ARAUCA',
       'POLIMODAL TELECAMPUS2_POLIMODAL', 'SOACHA', 'ITAGUI', 'ZIPAQUIRA',
       'CHIA CENTRO', 'BIZEN'], dtype=object)

In [28]:
for sede in lista_telecampus:

    mask = recibos_contact_1['PROPIETARIO_ZOHO'].str.contains(
        sede,
        case=False,
        na=False
    )

    recibos_contact_1.loc[
        mask,
        'PROPIETARIO_ZOHO'
    ] = f'TELECAMPUS {sede}'

In [29]:
#se crean las llaves para cruzar con tarifas
recibos_contact_1['llave_tarifas'] = recibos_contact_1['PROPIETARIO_ZOHO']+recibos_contact_1['NUEVO']+recibos_contact_1['TIPO']
recibos_contact_1['llave_tarifas']=recibos_contact_1['llave_tarifas'].str.upper()

tarifas_df['llave_tarifas']=tarifas_df['TELECAMPUS']+tarifas_df['TIPO']+tarifas_df['PRODUCTO']
tarifas_df['llave_tarifas']=tarifas_df['llave_tarifas'].str.upper()

In [30]:
# se cruzan tarifas con las ventas
df_recibos_tarifas = recibos_contact_1.merge(tarifas_df[['llave_tarifas','Por_COMISION1','Por_COMISION1_ING']], 
                                                on='llave_tarifas', 
                                                how='left')
##df_recibos_tarifas= recibos_contact_1[recibos_contact_1['llave_tarifas'].notna()]
df_recibos_tarifas.shape

(7, 27)

In [31]:
# SE RENOMBRAN COLUMNAS
df_recibos_tarifas.rename(columns={'PROPIETARIO_ZOHO': 'vendedor'}, inplace=True)
df_recibos_tarifas.rename(columns={'FUERZAZOHO': 'UNIDADNEGOCIO'}, inplace=True)
df_recibos_tarifas.rename(columns={'valor_ingles': 'VALOR_INGLES'}, inplace=True)
df_recibos_tarifas.rename(columns={'Por_COMISION1': 'PC_COMISION_MATRICULA'}, inplace=True)
df_recibos_tarifas.rename(columns={'Por_COMISION1_ING': 'PC_COMISION_INGLES'}, inplace=True)
df_recibos_tarifas.rename(columns={'FECHA_RECAUDO': 'FECHA_DE_RECAUDO'}, inplace=True)

#SE CREAN COLUMNAS
df_recibos_tarifas['MES_RECAUDO']=MES_RECAUDO_NUM
df_recibos_tarifas['PERIODO_FINANCIERO']=PERIODO_FINANCIERO
df_recibos_tarifas['MES_FACTURAR']=MES_FACTURAR

#------------TRATAMIENTO DE COLUMNAS NUMERICAS----------------------------
df_recibos_tarifas['VAL_LIQUIDADO'] = df_recibos_tarifas['VAL_LIQUIDADO'].fillna(0)
df_recibos_tarifas['VAL_LIQUIDADO'] = df_recibos_tarifas['VAL_LIQUIDADO'].astype(float)
df_recibos_tarifas['val_becdtos'] = df_recibos_tarifas['val_becdtos'].fillna(0)
df_recibos_tarifas['val_becdtos'] = df_recibos_tarifas['val_becdtos'].astype(float)
df_recibos_tarifas['RECARGO'] = df_recibos_tarifas['RECARGO'].fillna(0)
df_recibos_tarifas['RECARGO'] = df_recibos_tarifas['RECARGO'].astype(float)
df_recibos_tarifas['VALOR_INGLES'] = df_recibos_tarifas['VALOR_INGLES'].fillna(0)
df_recibos_tarifas['VALOR_INGLES'] = df_recibos_tarifas['VALOR_INGLES'].astype(float)
df_recibos_tarifas['ADICIONALES'] = df_recibos_tarifas['ADICIONALES'].fillna(0)
df_recibos_tarifas['ADICIONALES'] = df_recibos_tarifas['ADICIONALES'].astype(float)


#---------------- SE ELIMINAN COLUMNAS ------------------------------------
df_recibos_tarifas = df_recibos_tarifas.drop(columns=['PRODUCTO'])

df_recibos_tarifas.shape

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_24904\2469480841.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_recibos_tarifas['RECARGO'] = df_recibos_tarifas['RECARGO'].fillna(0)


(7, 29)

In [32]:
df_recibos_tarifas['VALOR_RECAUDO'] = df_recibos_tarifas['VALOR_RECAUDO'].astype(int)
df_total_telec = df_recibos_tarifas[df_recibos_tarifas['VALOR_RECAUDO'] > 0 ].copy()
df_total_telec.shape

(7, 29)

In [33]:
#----------CALCULO VALOR MATRICULA

df_total_telec['VALOR_MATRICULA'] = df_total_telec['VAL_LIQUIDADO'] + df_total_telec['val_becdtos'] + df_total_telec['RECARGO']

#------------CALCULO VALOR A PAGAR----------------------------

df_total_telec['VALOR_A_PAGAR'] = 0
condicion_preg_telec1=(df_total_telec['TIPO'].str.contains('PREGRADO', case=False, na=False))

# PARA PREGRADOS
df_total_telec.loc[condicion_preg_telec1,'VALOR_A_PAGAR'] = df_total_telec['VALOR_MATRICULA'] + df_total_telec['VALOR_INGLES'] + df_total_telec['ADICIONALES']

# NO PREGRADOS
df_total_telec.loc[~condicion_preg_telec1,'VALOR_A_PAGAR'] = df_total_telec['VALOR_MATRICULA'] + df_total_telec['ADICIONALES']

#------------CALCULO PESO MATRICULA---------------------------
df_total_telec['PESO_MATRICULA']= (df_total_telec['VALOR_MATRICULA']) / (df_total_telec['VALOR_A_PAGAR'])

#------------CALCULO PESO INGLES----------------------------
df_total_telec['PESO_INGLES'] = (df_total_telec['VALOR_INGLES']) / (df_total_telec['VALOR_A_PAGAR'])

#------------CALCULO PESO SEGURO----------------------------
df_total_telec['PESO_SEGURO'] = 1-(df_total_telec['PESO_MATRICULA']+df_total_telec['PESO_INGLES'])

#-----------PC PAGADO----------------------------
# Reemplaza ceros por NaN para evitar división por cero
val_deuda_sin_cero = df_total_telec['VALOR_A_PAGAR'].replace(0, np.nan)

# Haz la división y luego rellena NaN con 0 si deseas
df_total_telec['PC_PAGADO'] = (df_total_telec['VALOR_RECAUDO'] / val_deuda_sin_cero).fillna(0)
df_total_telec.shape

(7, 35)

In [34]:
df_total_telec['FECHA_DE_RECAUDO'] = pd.to_datetime(df_total_telec['FECHA_DE_RECAUDO'], dayfirst=True)

# Ahora si quieres que se guarde como string en formato AAAA-MM-DD:
df_total_telec['FECHA_DE_RECAUDO'] = df_total_telec['FECHA_DE_RECAUDO'].dt.strftime('%Y-%m-%d')

# SE TRAE LA BASE DE CL TIENE

In [35]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    
    base_cl_tiene = pd.read_sql(f'''SELECT IDENTIFICACION , PERIODO FROM CUN_REPOSITORIO.JUAN_GUZMAN.VISTA_GENERAL_CLTIENE
                                    WHERE PERIODO IN ({valores_unicos_periodos})''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [36]:
base_cl_tiene['LLAVE'] = base_cl_tiene['IDENTIFICACION'].astype(str)+base_cl_tiene['PERIODO'].astype(str)
base_cl_tiene['LLAVE'] = base_cl_tiene['LLAVE'].str.upper()
base_cl_tiene.shape

(89476, 3)

In [37]:
df_total_telec['CT_AYUDA'] = df_total_telec['LLAVE'].isin(base_cl_tiene['LLAVE']).astype(int)
df_total_telec.loc[df_total_telec['CT_AYUDA']==1,'CT_AYUDA'] = 'SI'
df_total_telec.loc[df_total_telec['CT_AYUDA']==0,'CT_AYUDA'] = 'NO'
df_total_telec.shape

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_24904\4267451118.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'SI' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_total_telec.loc[df_total_telec['CT_AYUDA']==1,'CT_AYUDA'] = 'SI'


(7, 36)

In [38]:
df_total_telec['SEGURO_Y_GASTOS'] = 0
df_total_telec.loc[df_total_telec['CT_AYUDA'] == 'SI', 'SEGURO_Y_GASTOS'] = 54000
df_total_telec.loc[df_total_telec['CT_AYUDA'] == 'NO', 'SEGURO_Y_GASTOS'] = 24000

# SE VALIDA QUE NO SE HAYA PAGADO LA COMISION

In [39]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    
    df_historico = pd.read_sql(f'''SELECT DOC_ALUM, PERIODO, FECHA_DE_RECAUDO, VALOR_RECAUDO
                               --, DEUDA_SEGURO_GASTOS 
                               FROM AGUEVARA.HISTORICO_COM_TELEC_2025''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [40]:
df_historico['DEUDA_SEGURO_GASTOS'] = '0'

In [41]:
# Se modifican los tipos de datos para que sean iguales entre dataframes
df_historico['FECHA_DE_RECAUDO'] = pd.to_datetime(df_historico['FECHA_DE_RECAUDO'], dayfirst=True)
df_historico['FECHA_DE_RECAUDO'] = df_historico['FECHA_DE_RECAUDO'].dt.strftime('%Y-%m-%d')
df_historico['VALOR_RECAUDO'] = df_historico['VALOR_RECAUDO'].astype(float).astype(int).astype(str)
df_total_telec['VALOR_RECAUDO'] = df_total_telec['VALOR_RECAUDO'].astype(float).astype(int).astype(str)

# Se crean las llaves
df_historico['LLAVE_PAGADO'] = df_historico['DOC_ALUM'].astype(str) + df_historico['PERIODO'] + df_historico['FECHA_DE_RECAUDO'] + df_historico['VALOR_RECAUDO']
df_historico['LLAVE_PAGADO'] = df_historico['LLAVE_PAGADO'].str.upper()
df_total_telec['LLAVE_PAGADO'] = df_total_telec['DOC_ALUM'].astype(str) + df_total_telec['PERIODO'] + df_total_telec['FECHA_DE_RECAUDO'] + df_total_telec['VALOR_RECAUDO']
df_total_telec['LLAVE_PAGADO'] = df_total_telec['LLAVE_PAGADO'].str.upper()
df_historico['LLAVE_SEGU_GAST'] = df_historico['DOC_ALUM'].astype(str) + df_historico['PERIODO']
df_historico['LLAVE_SEGU_GAST'] = df_historico['LLAVE_SEGU_GAST'].str.upper()

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_24904\3512413937.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_historico['FECHA_DE_RECAUDO'] = pd.to_datetime(df_historico['FECHA_DE_RECAUDO'], dayfirst=True)


In [42]:
df_total_telec['HISTORICO_RECAUDOS'] = (
    df_total_telec['LLAVE_PAGADO']
    .isin(df_historico['LLAVE_PAGADO'])
    .astype(int)
)

df_total_telec['HISTORICO_PERIODOS'] = (
    df_total_telec['LLAVE']
    .isin(df_historico['LLAVE_SEGU_GAST'])
    .astype(int)
)
df_total_telec.shape

(7, 40)

# SE DEJAN SOLO LAS VENTAS QUE NO SE HAYAN PAGADO

In [43]:
df_validos = df_total_telec[df_total_telec['HISTORICO_RECAUDOS'] == 0]
df_validos.shape

(7, 40)

In [44]:
df_validos = df_validos.merge(
    df_historico[['LLAVE_SEGU_GAST', 'DEUDA_SEGURO_GASTOS']],
    how='left',
    left_on='LLAVE',
    right_on='LLAVE_SEGU_GAST'
)

# Reemplazar nulos por 0
df_validos['DEUDA_SEGURO_GASTOS'] = (
    df_validos['DEUDA_SEGURO_GASTOS']
    .fillna(0)
)

df_validos['DEUDA_SEGURO_GASTOS'] = df_validos['DEUDA_SEGURO_GASTOS'].astype(int)
# Condición:
# si deuda = 0 y no cruzó con histórico
# entonces tomar valor de SEGURO_Y_GASTOS

df_validos.loc[
    (df_validos['DEUDA_SEGURO_GASTOS'] == 0) &
    (df_validos['HISTORICO_PERIODOS'] == 0),
    'DEUDA_SEGURO_GASTOS'
] = df_validos['SEGURO_Y_GASTOS']
df_validos

,DOC_ALUM,PERIODO_RECAUDO,VALOR_RECAUDO,FECHA_DE_RECAUDO,ORDEN,PERIODO,NUEVO,CLASE_ACTUAL,UNIDADNEGOCIO,ESTADO_PAGO,...,PESO_INGLES,PESO_SEGURO,PC_PAGADO,CT_AYUDA,SEGURO_Y_GASTOS,LLAVE_PAGADO,HISTORICO_RECAUDOS,HISTORICO_PERIODOS,LLAVE_SEGU_GAST,DEUDA_SEGURO_GASTOS
0,1034396370,2025D,691130,2026-08-03,3157394,2025D,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.065798,0.007567,0.227375,SI,54000,10343963702025D2026-08-03691130,0,1,10343963702025D,0
1,1034396370,2025D,691130,2026-08-03,3157394,2025D,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.065798,0.007567,0.227375,SI,54000,10343963702025D2026-08-03691130,0,1,10343963702025D,0
2,1034396370,2025D,691130,2026-08-03,3157394,2025D,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.065798,0.007567,0.227375,SI,54000,10343963702025D2026-08-03691130,0,1,10343963702025D,0
3,1152452884,25V01,615482,2026-08-17,2846769,25V01,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.161679,0.018593,0.497552,SI,54000,115245288425V012026-08-17615482,0,1,115245288425V01,0
4,80213480,24ET3,1779750,2026-08-25,0,24ET3,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.000000,0.006143,0.496929,NO,24000,8021348024ET32026-08-251779750,0,1,8021348024ET3,0
5,1022994511,25ES2,1,2026-08-14,3061370,23ET3,ANTIGUO,CONTINUO,PERMANENCIA,PAGO,...,0.000000,0.999952,0.000048,NO,24000,102299451123ET32026-08-141,0,0,NaN,24000
6,1013115057,25V05,432998,2026-08-06,3161375,25V05,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.054002,0.006210,0.116913,SI,54000,101311505725V052026-08-06432998,0,1,101311505725V05,0
7,1141317787,25V06,1198683,2026-08-20,3196911,25V03,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.183123,0.021059,1.097534,SI,54000,114131778725V032026-08-201198683,0,1,114131778725V03,0
8,1141317787,25V06,1198683,2026-08-20,3196911,25V03,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.183123,0.021059,1.097534,SI,54000,114131778725V032026-08-201198683,0,1,114131778725V03,0
9,1141317787,25V06,1198683,2026-08-20,3196911,25V03,NUEVO,NUEVO,TELECAMPUS,PAGO,...,0.183123,0.021059,1.097534,SI,54000,114131778725V032026-08-201198683,0,1,114131778725V03,0


# FUNCION PARA CALCULAR LA DEUDA RESTANTE DE SEGURO MEDICO Y GASTOS TECNICOS DEL ESTUDIANTE 

In [45]:
import numpy as np
import pandas as pd

# Convertir columnas a numéricas
df_validos['DEUDA_SEGURO_GASTOS'] = pd.to_numeric(
    df_validos['DEUDA_SEGURO_GASTOS'],
    errors='coerce'
).fillna(0)

df_validos['VALOR_RECAUDO'] = pd.to_numeric(
    df_validos['VALOR_RECAUDO'],
    errors='coerce'
).fillna(0)

# Función
def calcular_pago(deuda, recaudo):

    deuda_original = deuda

    # Si la deuda es menor o igual al recaudo
    if deuda <= recaudo:
        nueva_deuda = deuda

    else:
        nueva_deuda = deuda

        # Dividir hasta que sea menor o igual al recaudo
        while nueva_deuda > recaudo:
            nueva_deuda = nueva_deuda / 2

    # Resultados
    deuda_restante = deuda_original - nueva_deuda
    recaudo_restante = recaudo - nueva_deuda

    return pd.Series([
        round(nueva_deuda, 0),
        round(deuda_restante, 0),
        round(recaudo_restante, 0)
    ])

# Aplicar al dataframe
df_validos[[
    'NUEVA_DEUDA',
    'DEUDA_RESTANTE',
    'RECAUDO_RESTANTE'
]] = df_validos.apply(
    lambda x: calcular_pago(
        x['DEUDA_SEGURO_GASTOS'],
        x['VALOR_RECAUDO']
    ),
    axis=1
)

df_validos.shape

(11, 45)

In [46]:
df_validos_final = df_validos[['DOC_ALUM', 'PERIODO', 'TIPO',  'NOM_PROGRAMA','COD_UNI','UNIDADNEGOCIO','vendedor','VALOR_RECAUDO', 'FECHA_DE_RECAUDO','PERIODO_RECAUDO',
       'NUEVO', 'CLASE_ACTUAL', 'ESTADO_PAGO', 'EST_MATRICULADO',
       'ORDEN','valor_orden', 'VAL_LIQUIDADO', 'RECARGO',
       'ADICIONALES', 'val_becdtos', 'NCR_ANULACION', 'VALOR_INGLES', 'MES_RECAUDO', 'PERIODO_FINANCIERO', 'MES_FACTURAR', 
       'PC_COMISION_MATRICULA', 'PC_COMISION_INGLES',
       'VALOR_MATRICULA','VALOR_A_PAGAR', 'PESO_MATRICULA', 'PESO_INGLES', 'PESO_SEGURO',
       'PC_PAGADO', 'CT_AYUDA', 'SEGURO_Y_GASTOS','DEUDA_RESTANTE',
       'RECAUDO_RESTANTE']].copy()
df_validos_final = df_validos_final.rename(columns={'DEUDA_RESTANTE':'DEUDA_SEGURO_GASTOS','RECAUDO_RESTANTE':'VALOR_N_RECAUDO'})
df_validos_final.shape

(11, 37)

In [47]:
#CALCULO LA NUEVA COMISION------------------------------------------------------------
condicion_pregrado=(df_validos_final['TIPO'].str.contains('PREGRADO', case=False, na=False))

df_validos_final['COMISION_FINAL']= '0' 
# PARA PREGRADOS
df_validos_final.loc[condicion_pregrado,'COMISION_FINAL'] = ((df_validos_final['VALOR_N_RECAUDO'])*   
                                                               (df_validos_final['PC_COMISION_MATRICULA'])*
                                                               (df_validos_final['PESO_MATRICULA']))+((df_validos_final['VALOR_N_RECAUDO'])*
                                                                (df_validos_final['PC_COMISION_INGLES'])*(df_validos_final['PESO_INGLES']))
# NO PREGRADOS
df_validos_final.loc[~condicion_pregrado,'COMISION_FINAL'] = ((df_validos_final['VALOR_N_RECAUDO'])*(df_validos_final['PC_COMISION_MATRICULA'])*(df_validos_final['PESO_MATRICULA']))
#------------------------------
#------------------------------

In [48]:
df_validos_final['FINANCIADO'] = ((df_validos_final['VALOR_A_PAGAR']) == df_validos_final['VALOR_RECAUDO']).astype(int)
df_validos_final.loc[df_validos_final['FINANCIADO'] == 0, 'FINANCIADO'] ='SI'
df_validos_final.loc[df_validos_final['FINANCIADO'] == 1, 'FINANCIADO'] ='NO'

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_24904\2361733127.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'SI' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_validos_final.loc[df_validos_final['FINANCIADO'] == 0, 'FINANCIADO'] ='SI'


# SE ASIGNAN LOS NOMBRES A LOS ESTUDIANTES

In [49]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)
    
with engine.connect() as connection:
    try:
        sql_query = text(f''' 
            SELECT 
                F.IDENTIFICACION_BTPR,
                CONCAT(F.nombre_btpr, ' ', F.segundo_nombre_btpr, ' ', F.primer_apellido_btpr, ' ', F.segundo_apellido_btpr) AS NOMBRE_ESTUDIANTE
            FROM CUN_REPOSITORIO.DBO.HUB_PERSONA F
        ''')

        df_nombre = pd.read_sql_query(sql_query, connection)

    finally:
        connection.close()

Conexion a la base de datos realizada


In [50]:
# Suponiendo que 'IDENTIFICACION_BTPR' en df_nombres corresponde a 'DOC_ALUM' en df_historico
cedulas_validas = df_validos_final['DOC_ALUM'].unique()

df_nombres_filtrado = df_nombre[df_nombre['IDENTIFICACION_BTPR'].isin(cedulas_validas)]

In [51]:
df_nombres_filtrado = df_nombres_filtrado.drop_duplicates()
df_validos_final_final = df_validos_final.merge(df_nombres_filtrado, how='left' ,left_on='DOC_ALUM',right_on='IDENTIFICACION_BTPR')
df_validos_final_final = df_validos_final_final.drop_duplicates()
df_validos_final_final.drop(columns={'IDENTIFICACION_BTPR'},inplace=True)
df_validos_final_final.rename(columns={'NOMBRE_ESTUDIANTE':'NOMBRE'}, inplace=True)
df_validos_final_final .shape

(7, 40)

In [52]:
df_validos_final_final = df_validos_final_final[['DOC_ALUM', 'NOMBRE', 'PERIODO', 'TIPO', 'NOM_PROGRAMA', 'COD_UNI',
       'UNIDADNEGOCIO', 'vendedor', 'VALOR_RECAUDO', 'FECHA_DE_RECAUDO',
       'PERIODO_RECAUDO', 'NUEVO', 'CLASE_ACTUAL', 'ESTADO_PAGO',
       'EST_MATRICULADO', 'ORDEN', 'valor_orden', 'VAL_LIQUIDADO', 'RECARGO',
       'ADICIONALES', 'val_becdtos', 'NCR_ANULACION', 'VALOR_INGLES',
       'MES_RECAUDO', 'PERIODO_FINANCIERO', 'MES_FACTURAR',
       'PC_COMISION_MATRICULA', 'PC_COMISION_INGLES', 'VALOR_MATRICULA',
       'VALOR_A_PAGAR', 'PESO_MATRICULA', 'PESO_INGLES', 'PESO_SEGURO',
       'PC_PAGADO', 'CT_AYUDA', 'SEGURO_Y_GASTOS', 'DEUDA_SEGURO_GASTOS',
       'VALOR_N_RECAUDO', 'COMISION_FINAL', 'FINANCIADO']]

# SE CARGA LA LIQUIDACIÓN AL HISTORICO

In [53]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'HISTORICO_COM_TELEC_2026'
schema_name_2_12 = 'AGuevara'
# Send the data to sql server
df_validos_final_final.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

# SE UNIFICAN Y SE CARGAN TODOS LOS NO VALIDOS

In [54]:
df_no_validos = df_total_telec[df_total_telec['HISTORICO_RECAUDOS'] == 1]
df_no_validos['ESTADO_FINAL'] = 'YA SE PAGO / YA SE ENCUENTRA EN LA LIQUIDACION'
df_no_validos = df_no_validos.rename(columns={'vendedor':'PROPIETARIO_ZOHO','UNIDADNEGOCIO':'FUERZAZOHO'})
df_no_validos['TELECAMPUS DE NOVEDAD'] = df_no_validos['PROPIETARIO_ZOHO']
df_no_validos = df_no_validos[['DOC_ALUM', 'PERIODO', 'NUEVO', 'CLASE_ACTUAL', 'FUERZAZOHO',
       'ESTADO_PAGO', 'EST_MATRICULADO', 'TIPO', 'NOM_PROGRAMA',
       'PROPIETARIO_ZOHO', 'ESTADO_FINAL', 'TELECAMPUS DE NOVEDAD']]
df_no_validos.shape

(0, 12)

In [55]:
df_ventas['DOC_ALUM'] = df_ventas['DOC_ALUM'].astype(str)
df_r_recaudos_copy_copy['DOC_ALUM'] = df_r_recaudos_copy_copy['DOC_ALUM'].astype(float).astype(int).astype(str)
df_no_recaudo = df_ventas[~df_ventas['DOC_ALUM'].isin(df_r_recaudos_copy_copy['DOC_ALUM'])].copy()
df_no_recaudo['ESTADO_FINAL'] = 'NO SE EVIDENCIA PAGOS'

In [56]:
df_r_recaudos_copy['DOC_ALUM'] = df_r_recaudos_copy['DOC_ALUM'].astype(float).astype(int).astype(str)
df_no_fecha = df_ventas[~df_ventas['DOC_ALUM'].isin(df_r_recaudos_copy['DOC_ALUM'])].copy()
df_no_fecha = df_no_fecha[~df_no_fecha['DOC_ALUM'].isin(df_no_recaudo['DOC_ALUM'])].copy()
df_no_fecha['ESTADO_FINAL'] = 'NO SE EVIDENCIA RECAUDO EN EL MES EN LIQUIDACION'

In [57]:
no_validos = pd.concat([df_no_recaudo,df_no_fecha])
no_validos['TELECAMPUS DE NOVEDAD'] = no_validos['PROPIETARIO_ZOHO']
no_validos = no_validos[['DOC_ALUM', 'PERIODO', 'NUEVO', 'CLASE_ACTUAL', 'FUERZAZOHO',
       'ESTADO_PAGO', 'EST_MATRICULADO', 'TIPO', 'NOM_PROGRAMA',
       'PROPIETARIO_ZOHO', 'ESTADO_FINAL', 'TELECAMPUS DE NOVEDAD']]
no_validos.shape

(2732, 12)

In [58]:
no_validos = pd.concat([no_validos,df_no_validos])
no_validos.shape

(2732, 12)

# SE CARGAN A LA BASE LAS VENTAS QUE NO PASAN X RECAUDO O YA SE PAGARON

In [59]:
table_name_2_12 = 'VENTAS_NOVEDADES_NOVALIDAS_TELECAMPUS'
schema_name_2_12 = 'AGuevara'
# Send the data to sql server
no_validos.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

# SE TRAEN TODAS LAS VENTAS NO VALIDAS EN EL MES PARA UNIRLAS AL ARCHIVO

In [60]:
try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    
    no_pasaron = pd.read_sql(f'''SELECT * FROM AGUEVARA.VENTAS_NOVEDADES_NOVALIDAS_TELECAMPUS''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")
no_pasaron.shape        

Conexion a la base de datos realizada
Conexión cerrada


(8050, 12)

# SE UNIFICAN LA LIQUIDACION DEL MES EN LIQUIDACION CON LAS NOVEDADES VALIDAS

In [61]:
rr
si_pasaron = pd.concat([df_historico,df_validos_final])

NameError: name 'rr' is not defined

# SE ARMA EL ARCHIVO PARA ENVIAR A LIQUIDACION

In [ ]:
with pd.ExcelWriter(f'LIQUIDACION_NOVEDADES_SEPTIEMBRE_2026_5).xlsx', engine='openpyxl') as writer:
    df_validos_final.to_excel(writer, sheet_name='VALIDOS', index=False)
    no_pasaron.to_excel(writer, sheet_name='NO VALIDOS', index=False)